# 03 — Chunk vaccine từ DB



In [1]:
from pathlib import Path
import json
import os

ROOT = Path("..").resolve()
if not (ROOT / "knowledge").exists():
    ROOT = Path.cwd()
    if (ROOT / "vaxcare-ai" / "knowledge").exists():
        ROOT = ROOT / "vaxcare-ai"

OUT_PATH = ROOT / "knowledge" / "chunks" / "chunks.jsonl"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

env_path = ROOT / ".env"
if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

DB_HOST = os.getenv("DB_HOST", "127.0.0.1")
DB_PORT = int(os.getenv("DB_PORT", "3306"))
DB_USER = os.getenv("DB_USER", "root")
DB_PASSWORD = os.getenv("DB_PASSWORD", "")
DB_NAME = os.getenv("DB_NAME", "vaxcare_2026")

print("DB:", f"{DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}")
print("OUT:", OUT_PATH)


DB: root@127.0.0.1:3306/vaxcare_2026
OUT: D:\TTTN-VaxCare-2026\vaxcare-ai\knowledge\chunks\chunks.jsonl


In [2]:
import pymysql

conn = pymysql.connect(
    host=DB_HOST,
    port=DB_PORT,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME,
    charset="utf8mb4",
    cursorclass=pymysql.cursors.DictCursor,
)
print("Connected OK")


Connected OK


In [3]:
records = []

with conn.cursor() as cur:
    cur.execute(
        "SELECT vaccine_id, vaccine_name, manufacturer, target_disease, required_doses, dose_interval_days, description, status FROM vaccines"
    )
    vaccines = cur.fetchall()
    for v in vaccines:
        vid = v["vaccine_id"]
        text = (
            f"Vaccine: {v.get('vaccine_name')}\n"
            f"Nhà sản xuất: {v.get('manufacturer') or 'N/A'}\n"
            f"Phòng bệnh: {v.get('target_disease') or 'N/A'}\n"
            f"Số mũi: {v.get('required_doses')}\n"
            f"Khoảng cách (ngày): {v.get('dose_interval_days')}\n"
            f"Mô tả: {v.get('description') or ''}"
        ).strip()
        records.append({
            "chunk_id": f"db-vaccine_{vid}",
            "doc_id": "db-vaccine",
            "title": v.get("vaccine_name") or f"vaccine_{vid}",
            "text": text,
            "source_file": "mysql:vaccines",
            "tier": "B",
            "vaccine_ids": [vid],
            "page": None,
        })
    print("vaccines:", len(vaccines))

    try:
        cur.execute(
            "SELECT p.protocol_id, p.protocol_name, p.description, p.vaccine_id, v.vaccine_name FROM vaccination_protocols p LEFT JOIN vaccines v ON v.vaccine_id = p.vaccine_id"
        )
        protocols = cur.fetchall()
    except Exception as e:
        print("protocols skip:", e)
        protocols = []

    for p in protocols:
        pid = p["protocol_id"]
        text = (
            f"Phác đồ: {p.get('protocol_name')}\n"
            f"Vaccine: {p.get('vaccine_name') or p.get('vaccine_id')}\n"
            f"Mô tả: {p.get('description') or ''}"
        ).strip()
        records.append({
            "chunk_id": f"db-protocol_{pid}",
            "doc_id": "db-protocol",
            "title": p.get("protocol_name") or f"protocol_{pid}",
            "text": text,
            "source_file": "mysql:vaccination_protocols",
            "tier": "B",
            "vaccine_ids": [p["vaccine_id"]] if p.get("vaccine_id") else [],
            "page": None,
        })
    print("protocols:", len(protocols))

    try:
        cur.execute(
            "SELECT d.detail_id, d.protocol_id, d.dose_number, d.age_in_months, d.interval_days, d.notes FROM protocol_details d ORDER BY d.protocol_id, d.dose_number"
        )
        details = cur.fetchall()
    except Exception as e:
        print("protocol_details skip:", e)
        details = []

    for d in details:
        did = d["detail_id"]
        text = (
            f"Protocol detail #{did} (protocol_id={d.get('protocol_id')})\n"
            f"Mũi số: {d.get('dose_number')}\n"
            f"Tuổi (tháng): {d.get('age_in_months')}\n"
            f"Khoảng cách (ngày): {d.get('interval_days')}\n"
            f"Ghi chú: {d.get('notes') or ''}"
        ).strip()
        records.append({
            "chunk_id": f"db-protocol-detail_{did}",
            "doc_id": "db-protocol-detail",
            "title": f"protocol_{d.get('protocol_id')}_dose_{d.get('dose_number')}",
            "text": text,
            "source_file": "mysql:protocol_details",
            "tier": "B",
            "vaccine_ids": [],
            "page": None,
        })
    print("details:", len(details))

conn.close()

mode = "a" if OUT_PATH.exists() else "w"
with OUT_PATH.open(mode, encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Exported {len(records)} DB chunks -> {OUT_PATH} (mode={mode})")


vaccines: 13
protocols: 14
protocol_details skip: (1054, "Unknown column 'd.detail_id' in 'field list'")
details: 0
Exported 27 DB chunks -> D:\TTTN-VaxCare-2026\vaxcare-ai\knowledge\chunks\chunks.jsonl (mode=a)
